# epiTOC2

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)


Let's first import some packages:

In [30]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [31]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.epiTOC2)


class epiTOC2(pyagingModel):
    def __init__(self):
        super().__init__()
        self.delta = None
        self.beta0 = None

    def preprocess(self, x):
        """
        Replace NaNs with zero; missing features should already be imputed via reference_values.
        """
        return torch.nan_to_num(x, nan=0.0)

    def forward(self, x):
        x = self.preprocess(x)

        device = x.device
        dtype = x.dtype

        delta = self.delta.to(device=device, dtype=dtype)
        beta0 = self.beta0.to(device=device, dtype=dtype)

        denom = delta * (1 - beta0)
        denom = torch.where(denom == 0, torch.ones_like(denom), denom)

        contrib = (x - beta0) / denom
        k = contrib.size(1)
        vals = 2.0 * torch.sum(contrib, dim=1) / k

        return self.postprocess(vals.unsqueeze(1))

    def postprocess(self, x):
        return x



In [32]:
model = pya.models.epiTOC2()

## Define clock metadata

In [33]:
model.metadata["clock_name"] = "epitoc2"
model.metadata["data_type"] = "DNA methylation"  # Paper: The study constructs clocks from DNA methylation measurements.
model.metadata["species"] = "Homo sapiens"  # Paper: The analyzed samples and clock are human.
model.metadata["year"] = 2020
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Teschendorff, Andrew E. \"A comparison of epigenetic mitotic-like clocks for cancer risk prediction.\" Genome Medicine 12 (2020): 56."
model.metadata["doi"] = "https://doi.org/10.1186/s13073-020-00752-3"
model.metadata["notes"] = "Dynamic methylation-transmission model returning total cumulative stem-cell divisions per stem cell; an intrinsic rate additionally requires chronological age but is not this implementation's returned value."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: Parameters were fitted in the Hannum whole-blood cohort.
model.metadata["predicts"] = ["mitotic age"]  # Paper: The epiTOC2 estimate is total stem-cell divisions per stem cell.
model.metadata["training_target"] = ["chronological age"]  # Paper: CpG dynamic-model parameters were fitted against age-associated methylation trajectories.
model.metadata["unit"] = ["cell divisions per stem cell"]  # Paper: The model output is total stem-cell divisions per stem cell.
model.metadata["model_type"] = "dynamic methylation transmission model"  # Paper: A dynamic model of methylation transmission with stem-cell divisions defines epiTOC2.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: The Hannum training cohort was measured on the Illumina 450K array.
model.metadata["population"] = "adults"  # Paper: Model fitting used Hannum whole blood spanning ages 19 to 101.
model.metadata["journal"] = "Genome Medicine"
model.metadata["last_author"] = "Andrew E. Teschendorff"
model.metadata["n_features"] = 163
model.metadata["citations"] = 155
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

#### Download coefficient file

In [34]:
coeff_url = "https://raw.githubusercontent.com/bio-learn/biolearn/master/biolearn/data/EpiTOC2.csv"
os.system(f"curl -L {coeff_url} -o EpiTOC2.csv")


0

## Load features

#### From Excel file

In [35]:
df = pd.read_csv('EpiTOC2.csv', index_col=0)
df['feature'] = df.index.astype(str)
model.features = df['feature'].tolist()


## Load weights into base model


In [36]:
#### From CSV file


In [37]:
model.delta = torch.tensor(df['delta'].values, dtype=torch.float32).unsqueeze(0)
model.beta0 = torch.tensor(df['beta0'].values, dtype=torch.float32).unsqueeze(0)
model.base_model = None


In [38]:
#### Linear model


# Not used; computation happens in the model forward


In [39]:
model.reference_values = [-1]*len(model.features)

model.reference_values = [0.0] * len(model.features)

In [40]:
model.preprocess_name = "mean"
model.preprocess_dependencies = None

In [41]:
model.preprocess_name = "nan_to_zero"
model.preprocess_dependencies = None


model.postprocess_name = None
model.postprocess_dependencies = None


In [42]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Teschendorff, Andrew E. "A comparison of epigenetic mitotic-like '
             'clocks for cancer risk prediction." Genome Medicine 12.1 (2020): '
             '56.',
 'clock_name': 'epitoc2',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1186/s13073-020-00752-3',
 'notes': 'Stem cell division rate estimate using EpiTOC2.',
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2020}
reference_values: [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]... [Total elements: 163]
preprocess_name: 'nan_to_zero'
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg00043095', 'cg00347369', 'cg00397986', 'cg00466268', 'cg00884606', 'cg00916884', 'cg01435574', 'cg01537995'

## Normal feature ranges

Units and plausible bounds come from `pyaging`'s feature range registry, keyed by feature name with a fallback to the default for the clock's `data_type`. `predict_age` warns when input values fall outside these bounds, which usually means the data is in different units than the clock expects.

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [44]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory

In [45]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: EpiTOC2.csv
